# TP 2 — CNN sur Fashion MNIST et augmentation de données

**Objectifs**

- Travailler sur Fashion MNIST (10 classes de vêtements 28×28 grayscale, plus complexe que MNIST chiffres).
- Construire un CNN à 3 blocs convolutifs.
- Comparer entraînement avec et sans augmentation de données.

**Durée indicative : 60 minutes.**

> Note : on utilise un sous-ensemble (5 000 train / 1 000 test) pour rester en quelques minutes sur CPU. L'objectif est pédagogique, pas la performance absolue. Le premier appel téléchargera Fashion MNIST (~30 Mo) dans `data/`.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

torch.manual_seed(0)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DATA_ROOT = "data"
FASHION_CLASSES = [
    "T-shirt/top",
    "Pantalon",
    "Pull",
    "Robe",
    "Manteau",
    "Sandale",
    "Chemise",
    "Sneaker",
    "Sac",
    "Bottine",
]
# Stats canal unique (grayscale) calculees sur le train Fashion MNIST.
MEAN, STD = (0.2860,), (0.3530,)

## Exercice 1 — Charger Fashion MNIST sans augmentation

1. Crée un transform `Compose([ToTensor, Normalize(MEAN, STD)])`.
2. Charge Fashion MNIST train et test, prends 5 000 / 1 000 premiers exemples.
3. Crée les DataLoaders (batch 128 train, 256 test).
4. Affiche 10 exemples (en re-dénormalisant pour avoir une image lisible) avec leur classe.

<details>
<summary>💡 Coup de pouce</summary>

- `MEAN = (0.2860,)`, `STD = (0.3530,)` : tuple à 1 élément car **1 seul canal** (grayscale).
- Pour ré-afficher après normalisation :
  ```python
  img = img * STD[0] + MEAN[0]
  img = img.clamp(0, 1).squeeze()         # retire le canal pour matplotlib
  plt.imshow(img, cmap="gray")
  ```
- ⚠️ **`cmap="gray"` est obligatoire** : sans ça, matplotlib utilise la colormap par défaut (viridis) sur des images grayscale et l'affichage est trompeur.

</details>


In [ ]:
# TODO

## Exercice 2 — Construire un CNN à 3 blocs

Architecture suggérée (inspirée VGG simplifié, adaptée au 1×28×28) :

- Conv(1→32, k=3, padding=1) → ReLU → Conv(32→32) → ReLU → Pool 2×2
- Conv(32→64) → ReLU → Conv(64→64) → ReLU → Pool 2×2
- Conv(64→128) → ReLU → Pool 2×2
- Flatten → Linear(128*3*3 → 128) → ReLU → Dropout(0.5) → Linear(128 → 10)

Vérifie que `model(torch.zeros(1, 1, 28, 28)).shape == (1, 10)`.

<details>
<summary>💡 Coup de pouce</summary>

- **Calcul des dimensions** : entrée 28×28 → pool → 14×14 → pool → 7×7 → pool → 3×3 (`MaxPool2d(2)` fait un floor : 7//2 = 3).
- Donc après le dernier bloc, la sortie spatiale est **3×3** avec 128 canaux.
- FC d'entrée : `nn.Linear(128 * 3 * 3, 128)` car 128 × 3 × 3 = 1152.
- ⚠️ **Différence avec CIFAR-10** : ici `in_channels=1` (grayscale) et la dim finale est 3×3 et non 4×4. Adapter ces deux nombres dans votre code.
- Dropout : `nn.Dropout(0.5)` se met **avant** la dernière FC.

</details>


In [ ]:
# TODO

## Exercice 3 — Entraînement sans augmentation

Réutilise tes fonctions `train_one_epoch` et `evaluate` du TP 1 (ou recopie-les). Entraîne 3 époques avec `Adam(lr=1e-3)`. Conserve l'historique des accuracies.


In [ ]:
# TODO

## Exercice 4 — Avec augmentation de données

Crée un transform train avec augmentation :

```python
train_aug = transforms.Compose([
    transforms.RandomCrop(28, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])
```

Recharge le dataset train avec ce transform (le test ne change pas). Ré-instancie un modèle frais, entraîne 3 époques. Compare les courbes accuracy avec et sans augmentation.

<details>
<summary>💡 Coup de pouce</summary>

- Crucial : seul le **train** utilise les augmentations, le **test** reste avec un transform basique (ToTensor + Normalize).
- Recharge le dataset avec le nouveau `train_aug` : `train = datasets.FashionMNIST(root, train=True, transform=train_aug, download=True)`.
- `RandomHorizontalFlip` est légitime sur Fashion MNIST : les vêtements/sacs/chaussures sont quasi-symétriques. À l'inverse, on évite ce flip sur des datasets de chiffres (un 6 flippé devient un truc bizarre).
- Tu devrais voir l'accuracy test monter plus haut (+1-3 pts) avec augmentation, surtout sur peu d'époques.

</details>


In [ ]:
# TODO